In [0]:
dbutils.widgets.text("user", "")
dbutils.widgets.text("pw", "")
dbutils.widgets.text("window_days", "6")
dbutils.widgets.text("window_minutes", "5")
dbutils.widgets.text("max_windows", "0")   # 0 = run to completion

SQL_SERVER = "ecommproject-sqlserver.database.windows.net"
SQL_DB     = "ecommproject-sqldb"
EVENTS     = "abfss://basecontainer@ecommprojadls.dfs.core.windows.net/order_events_history/events.parquet"
START      = "2017-10-01"

WINDOW_DAYS    = int(dbutils.widgets.get("window_days"))
WINDOW_MINUTES = int(dbutils.widgets.get("window_minutes"))
MAX_WINDOWS    = int(dbutils.widgets.get("max_windows"))

JDBC = (f"jdbc:sqlserver://{SQL_SERVER}:1433;database={SQL_DB};"
        f"user={dbutils.widgets.get('user')};password={dbutils.widgets.get('pw')};"
        "encrypt=true;trustServerCertificate=false;loginTimeout=60;")

In [0]:
import pandas as pd

ev = spark.read.parquet(EVENTS).toPandas()
START_TS = pd.Timestamp(START)
ev["w"] = ((ev.t - START_TS).dt.total_seconds() // (WINDOW_DAYS * 86400)).astype(int)
TOTAL = int(ev.w.max()) + 1

print(f"{len(ev):,} events across {TOTAL} windows")
print(f"est. runtime: {TOTAL * WINDOW_MINUTES / 60:.1f} hrs")
print(ev.groupby("w").size().describe().to_string())

In [0]:
import time
from datetime import datetime

jvm = spark.sparkContext._gateway.jvm

def nn(v):
    """NaT/NaN → SQL NULL. Timestamps go across as ISO strings; the JDBC
    driver parses them into datetime2 correctly and this sidesteps
    Python↔Java date object conversion entirely."""
    return None if pd.isna(v) else str(v)

def get_conn():
    c = jvm.java.sql.DriverManager.getConnection(JDBC)
    c.setAutoCommit(False)   # we control the transaction boundary per window
    return c

def batch(conn, sql, rows):
    """One prepared statement, one round trip. Row-at-a-time would be
    ~30ms each — a 3,000-row window would take 90 seconds instead of 2."""
    if not rows:
        return 0
    ps = conn.prepareStatement(sql)
    for r in rows:
        for i, v in enumerate(r, start=1):
            ps.setObject(i, v)
        ps.addBatch()
    ps.executeBatch()
    ps.close()
    return len(rows)

In [0]:
conn = get_conn()

rs = conn.createStatement().executeQuery(
    "SELECT ISNULL(MAX(last_window_index), -1) FROM dbo.loader_checkpoint")
rs.next()
start_w = rs.getInt(1) + 1
limit = TOTAL if MAX_WINDOWS == 0 else min(TOTAL, start_w + MAX_WINDOWS)

print(f"resuming at window {start_w}, running to {limit - 1}")

for w in range(start_w, limit):
    t0 = time.monotonic()
    g = {k: v.sort_values("t") for k, v in ev[ev.w == w].groupby("kind")}
    sim = START_TS + pd.Timedelta(days=WINDOW_DAYS * (w + 1))
    n = {}

    try:
        if "order_insert" in g:
            n["oi"] = batch(conn,
                "INSERT INTO dbo.orders (order_id, customer_unique_id, order_status,"
                " order_purchase_timestamp, order_estimated_delivery_date, updated_at)"
                " VALUES (?,?,?,?,?,?)",
                [(r.order_id, r.customer_unique_id, r.status, str(r.t),
                  nn(r.estimated), str(r.t)) for r in g["order_insert"].itertuples()])

        if "item_insert" in g:
            n["ii"] = batch(conn,
                "INSERT INTO dbo.order_items (order_id, order_item_id, product_id,"
                " seller_id, shipping_limit_date, price, freight_value, updated_at)"
                " VALUES (?,?,?,?,?,?,?,?)",
                [(r.order_id, int(r.order_item_id), r.product_id, r.seller_id,
                  nn(r.shipping_limit_date), float(r.price), float(r.freight_value),
                  str(r.t)) for r in g["item_insert"].itertuples()])

        if "order_update" in g:
            n["ou"] = batch(conn,
                "UPDATE dbo.orders SET order_status=?, order_approved_at=?,"
                " order_delivered_carrier_date=?, order_delivered_customer_date=?,"
                " updated_at=? WHERE order_id=?",
                [(r.status, nn(r.approved_at), nn(r.carrier_date),
                  nn(r.customer_date), str(r.t), r.order_id)
                 for r in g["order_update"].itertuples()])

        if "item_update" in g:
            n["iu"] = batch(conn,
                "UPDATE dbo.order_items SET freight_value=?, updated_at=?"
                " WHERE order_id=? AND order_item_id=?",
                [(float(r.freight_value), str(r.t), r.order_id, int(r.order_item_id))
                 for r in g["item_update"].itertuples()])

        if "customer_move" in g:
            n["cm"] = batch(conn,
                "UPDATE dbo.customers SET customer_city=?, customer_state=?,"
                " customer_zip_prefix=?, updated_at=? WHERE customer_unique_id=?",
                [(r.customer_city, r.customer_state, r.customer_zip_prefix,
                  str(r.t), r.customer_unique_id)
                 for r in g["customer_move"].itertuples()])

        if "order_delete" in g:
            n["od"] = batch(conn, "DELETE FROM dbo.orders WHERE order_id=?",
                [(r.order_id,) for r in g["order_delete"].itertuples()])

        batch(conn,
            "MERGE dbo.loader_checkpoint AS t USING (SELECT 1 id) s ON t.id=s.id"
            " WHEN MATCHED THEN UPDATE SET last_window_index=?, last_sim_time=?,"
            " updated_at=SYSUTCDATETIME()"
            " WHEN NOT MATCHED THEN INSERT (id, last_window_index, last_sim_time)"
            " VALUES (1,?,?);",
            [(w, str(sim), w, str(sim))])

        conn.commit()
    except Exception:
        conn.rollback()
        print(f"window {w} failed and rolled back — rerun resumes here")
        raise

    el = time.monotonic() - t0
    print(f"[{datetime.utcnow():%H:%M:%S}] w{w:>3}/{TOTAL-1} sim<{sim:%Y-%m-%d} "
          f"{el:5.1f}s {n}", flush=True)

    pause = WINDOW_MINUTES * 60 - el
    if pause > 0 and w < limit - 1:
        time.sleep(pause)

conn.close()
print("replay complete")